# TriNetra-AMRF — M3FD Four-Way TRC Ablation (Colab)

Fast comparative pass — RGB-only / Thermal-only / Fusion-no-TRC / Fusion-with-TRC —
on a data slice of the M3FD dataset, run on Colab's GPU (typically a T4, 16GB VRAM,
well above this project's usual 6GB laptop GPU).

**Before running:** zip your local `datasets/M3FD/` folder (the `Vis/`, `Ir/`,
`Annotation/` subfolders) and upload it to Google Drive — e.g. `MyDrive/M3FD.zip`.
It is *not* committed to the GitHub repo (too large), so it has to come in via Drive.

**Runtime:** Runtime -> Change runtime type -> GPU (T4 is fine) before running cells.

## 1. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repo

In [ ]:
!git clone https://github.com/SadiqueK78/Tri-Netra.git
%cd Tri-Netra

## 3. Install dependencies

Colab already ships PyTorch+CUDA and most scientific packages — only the
project-specific ones are installed here. `albumentations` is pinned to
1.4.6: newer versions pull in `albucore`->`stringzilla`, which has no
prebuilt wheel for some platforms and needs a C compiler to build from
source (this bit us hard on Windows; pinning sidesteps it everywhere).

In [ ]:
!pip install -q "albumentations==1.4.6" --no-deps
!pip install -q ultralytics deep-sort-realtime pycocotools python-dotenv GPUtil nvidia-ml-py

## 4. Mount Drive and unpack the M3FD dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Edit this path to wherever you uploaded the zip in your Drive.
M3FD_ZIP = "/content/drive/MyDrive/M3FD.zip"

!mkdir -p datasets/M3FD
!unzip -q "$M3FD_ZIP" -d datasets/M3FD
# If the zip contains a top-level M3FD/ folder itself, flatten it one level:
!if [ -d datasets/M3FD/M3FD ]; then mv datasets/M3FD/M3FD/* datasets/M3FD/ && rmdir datasets/M3FD/M3FD; fi
!echo '--- expect Vis/ Ir/ Annotation/ below ---' && ls datasets/M3FD

## 5. Convert annotations and build the split

Same scripts already validated locally — `configs/default.yaml` in this
repo already has `active_source: m3fd` wired up from that work.

In [ ]:
!python datasets/convert_annotations.py --source m3fd --validate

In [ ]:
!python datasets/split_dataset.py --config configs/default.yaml

## 6. Pre-flight checks

Cheap sanity checks before spending GPU time — same ones run locally
before every training pass in this project.

In [ ]:
!python -m training.loss_adapter --self-test

In [ ]:
!python utils/check_fusion.py --test loss --split val

## 7. Run the four-way comparative ablation

800 train / 200 held-out val images per variant, 8 epochs — the same
scale as the LLVIP comparative pass this project already ran. Colab's GPU
has much more headroom than the 6GB laptop GPU this was developed on, so
feel free to raise `--subset` / `--epochs` if there's time before the
presentation (e.g. `--subset 2000 --epochs 15`) for stronger numbers.

In [ ]:
!python training/run_ablation.py --subset 800 --epochs 8

## 8. Show the results table

In [ ]:
from IPython.display import Markdown, display
with open('runs/ablation/results.md') as f:
    display(Markdown(f.read()))

## 9. Demo: run inference on one pair and show the detection image

Good for the presentation itself — a real annotated detection, not just a metrics table.

In [ ]:
import glob
sample_vis = sorted(glob.glob('datasets/visible/test/*'))[0]
sample_name = sample_vis.split('/')[-1]
sample_thr = f'datasets/thermal/test/{sample_name}'
print('visible:', sample_vis)
print('thermal:', sample_thr)

!python inference/predict_image.py \
    --checkpoint weights/ablation/poc/fusion_trc/best.pt \
    --visible "$sample_vis" --thermal "$sample_thr" \
    --output runs/predictions/colab_demo.jpg

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage('runs/predictions/colab_demo.jpg'))

## 10. Save everything back to Drive

Colab's local disk is wiped when the runtime disconnects — copy the
checkpoints, results table, and demo image to Drive before closing.

In [ ]:
import os
OUT_DIR = "/content/drive/MyDrive/TriNetra_M3FD_results"
os.makedirs(OUT_DIR, exist_ok=True)
!cp -r runs/ablation "$OUT_DIR/"
!cp -r weights/ablation "$OUT_DIR/"
!cp runs/predictions/colab_demo.jpg "$OUT_DIR/"
print('Saved to', OUT_DIR)